# Qwen3.5-2B CBSE Bengali: 6-Dimension LLM-as-Judge (0-4)

Evaluates **one Qwen generation per Gemini API call** against its question + reference answer.

Each input row contains: `id`, `subject`, `grade`, `district`, `topic`, `context`, `question`, `reference_answer`, `generation` (+`finish_reason`, `truncated`).

**Dimensions (0-4 each, 4 = best):** groundedness, faithfulness, hallucination (4 = none), bengali_quality (fluency only), answer_relevancy, instruction_following.
`final_score` = round(mean of 6 dims, 2), computed in code. `reason` is empty iff all six are 4, else required.

Rows with `truncated=true` are auto-marked 0 without spending an API call. `context` is shown to the judge for local-detail checks.


In [ ]:
!pip install -q google-genai pandas tqdm


## 1. Configuration

Same model and conservative rate limits as the SARG judge notebook. Batch size is intentionally **1**: one generation = one API request.

In [ ]:
import os
import json
import time
from collections import deque
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from google import genai
from google.genai import types

# -----------------------------
# Model / files
# -----------------------------
MODEL_NAME = "gemini-3.5-flash-lite"
INPUT_FILE = "/content/qwen3p5_2b_generations_qonly_cbse_bengali_150.jsonl"
OUTPUT_FILE = "/content/qwen_judge_6dim_results.jsonl"

# -----------------------------
# One row per API call
# -----------------------------
ROWS_PER_REQUEST = 1

# Same conservative working limits as SARG judge notebook
RPM_HARD = 15
RPM_TARGET = 12
TPM_HARD = 250_000
TPM_TARGET = 200_000
RPD_HARD = 500

# Small delay helps avoid bursting requests.
MIN_REQUEST_INTERVAL = 60 / RPM_TARGET

print("Model:", MODEL_NAME)
print("Rows per request:", ROWS_PER_REQUEST)
print("RPM target:", RPM_TARGET)
print("TPM target:", TPM_TARGET)
print("RPD hard limit:", RPD_HARD)


## 2. API key

Set `GEMINI_API_KEY` in Colab Secrets/environment, then run this cell.

In [ ]:
API_KEY = os.environ.get("GEMINI_API_KEY")

if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass

if not API_KEY:
    raise RuntimeError("GEMINI_API_KEY not found. Add it to Colab Secrets or set the environment variable.")

client = genai.Client(api_key=API_KEY)
print("Gemini client initialized.")


## 3. Load and validate the generations file

`context` is required: the judge uses it for local-detail checks. `truncated` flags rows for auto-marking.

In [ ]:
REQUIRED_KEYS = [
    "id",
    "subject",
    "grade",
    "district",
    "topic",
    "context",
    "question",
    "reference_answer",
    "generation",
]

rows = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError as e:
            raise ValueError(f"Invalid JSON on line {line_no}: {e}")

        missing = [k for k in REQUIRED_KEYS if k not in row]
        if missing:
            raise ValueError(f"Line {line_no} is missing keys: {missing}")

        if not row["generation"].strip():
            raise ValueError(f"Line {line_no} ({row["id"]}) has an empty generation.")

        rows.append(row)

print(f"Loaded {len(rows)} rows.")
print("First row keys:", list(rows[0].keys()) if rows else "No rows")

if len(rows) > RPD_HARD:
    raise ValueError(f"Dataset has {len(rows)} rows, exceeding the {RPD_HARD} request hard limit.")

ids = [r["id"] for r in rows]
if len(ids) != len(set(ids)):
    raise ValueError("Duplicate IDs found.")

print("Unique IDs: OK")
auto_rows = [r for r in rows if r.get("truncated") is True]
judge_rows_todo = [r for r in rows if r.get("truncated") is not True]
print(f"Auto-mark (truncated, no API call): {len(auto_rows)}")
print(f"Planned API calls: {len(judge_rows_todo)}")


## 4. LLM judge rubric (6 dimensions, 0-4)

The judge receives the **entire row** (context + question + reference + generation) in one request.

### Scores (0-4 each, 4 = best)
- **Groundedness:** claims in the generation are supported by the reference answer.
- **Faithfulness:** nothing contradicts or distorts the reference answer.
- **Hallucination:** no invented facts/entities (4 = none at all, 0 = severe fabrication).
- **Bengali quality:** fluent, grammatical Bengali script. Arabic numerals are NOT penalized.
- **Answer relevancy:** actually answers the question asked.
- **Instruction following:** does the requested task (worksheet / explanation / feedback / etc.).

### Reason rule
- All six are 4 → `reason` must be `""` (empty).
- Any dim below 4 → `reason` must be 1-3 sentences naming the failed dims.

`final_score` is NOT assigned by the judge: the pipeline computes `round(mean(6 dims), 2)`.
Do not penalize different wording if the answer is correct. Repetition loops that restate without answering score ~0 on relevancy/instruction-following.

In [ ]:
JUDGE_PROMPT_TEMPLATE = """
You are an expert evaluator of Bengali educational AI answers (CBSE, West Bengal).

Score the GENERATION from 0-4 on each dimension using the CONTEXT, QUESTION and REFERENCE ANSWER.

1. Groundedness: are the generation's claims supported by the reference answer? (4 = all supported, 0 = mostly unsupported)
2. Faithfulness: does anything contradict or distort the reference answer? (4 = fully consistent, 0 = contradicts it)
3. Hallucination: are there invented facts, entities or events absent from both reference and question? (4 = none at all, 0 = severe fabrication)
4. Bengali quality: fluent, grammatical Bengali script? (4 = fully fluent; do NOT penalize Arabic numerals)
5. Answer relevancy: does it actually answer the question asked? (4 = directly answers, 0 = ignores it; repetition loops that restate without answering score 0-1)
6. Instruction following: does it do the requested task (worksheet / explanation / feedback / project guide)? (4 = exactly the task, 0 = wrong task)

Use CONTEXT (subject/grade/topic/district) to check whether local details are used meaningfully and consistently; do not demand verbatim reuse.
Do not penalize different wording if the answer is correct.

Reason rule (strict): if ALL six scores are 4, return reason as an empty string. If ANY score is below 4, return 1-3 sentences naming each failed dimension and why. No other text.

ID: {id}
Subject: {subject} | Grade: {grade} | District: {district} | Topic: {topic}
CONTEXT:
{context}

QUESTION:
{question}

REFERENCE ANSWER:
{reference_answer}

GENERATION:
{generation}

Return ONLY valid JSON:

{{\n  "groundedness": 0,\n  "faithfulness": 0,\n  "hallucination": 0,\n  "bengali_quality": 0,\n  "answer_relevancy": 0,\n  "instruction_following": 0,\n  "reason": ""\n}}\n"""

DIMENSIONS = [
    "groundedness",
    "faithfulness",
    "hallucination",
    "bengali_quality",
    "answer_relevancy",
    "instruction_following",
]

AUTO_REASON_TRUNCATED = (
    "Generation truncated (finish_reason=length); response cut off before completion, "
    "not a complete answer."
)

def compute_final_score(scores):
    """Mean of the six 0-4 dims, rounded to 2 decimals. Computed in code, never by the judge."""
    return round(sum(scores[d] for d in DIMENSIONS) / len(DIMENSIONS), 2)

def reason_ok(result):
    """Reason must be empty iff all six dims are 4."""
    all_four = all(result[d] == 4 for d in DIMENSIONS)
    has_reason = bool(result.get("reason", "").strip())
    return (not has_reason) if all_four else has_reason


## 5. Rate limiter

Identical to the SARG judge notebook: target 12 requests/minute with a rolling-minute token budget.

In [ ]:
CHARS_PER_TOKEN_ESTIMATE = 4

request_events = deque()
token_events = deque()
daily_requests = 0
last_request_time = 0.0

def estimate_tokens(text):
    return max(1, int(len(text) / CHARS_PER_TOKEN_ESTIMATE))

def prune_events(now):
    while request_events and now - request_events[0] >= 60:
        request_events.popleft()
    while token_events and now - token_events[0][0] >= 60:
        token_events.popleft()

def current_tokens():
    return sum(tokens for _, tokens in token_events)

def wait_for_rate_limit(estimated_tokens):
    global daily_requests, last_request_time

    if daily_requests >= RPD_HARD:
        raise RuntimeError("Daily request hard limit reached.")

    while True:
        now = time.time()
        prune_events(now)

        wait_time = max(0, MIN_REQUEST_INTERVAL - (now - last_request_time))

        if len(request_events) >= RPM_TARGET:
            wait_time = max(wait_time, 60 - (now - request_events[0]))

        if current_tokens() + estimated_tokens > TPM_TARGET and token_events:
            wait_time = max(wait_time, 60 - (now - token_events[0][0]))

        if wait_time <= 0:
            break

        time.sleep(wait_time)

    now = time.time()
    request_events.append(now)
    token_events.append((now, estimated_tokens))
    daily_requests += 1
    last_request_time = now


## 6. Gemini request helper

Response is forced toward JSON. The reason rule is validated after parsing; one retry is allowed on violation.

In [ ]:
JUDGE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "groundedness": {"type": "INTEGER"},
        "faithfulness": {"type": "INTEGER"},
        "hallucination": {"type": "INTEGER"},
        "bengali_quality": {"type": "INTEGER"},
        "answer_relevancy": {"type": "INTEGER"},
        "instruction_following": {"type": "INTEGER"},
        "reason": {"type": "STRING"},
    },
    "required": [
        "groundedness", "faithfulness", "hallucination",
        "bengali_quality", "answer_relevancy", "instruction_following",
        "reason",
    ],
}

def build_prompt(row):
    return JUDGE_PROMPT_TEMPLATE.format(
        id=row["id"],
        subject=row.get("subject", ""),
        grade=row.get("grade", ""),
        district=row.get("district", ""),
        topic=row.get("topic", ""),
        context=row.get("context", ""),
        question=row["question"],
        reference_answer=row["reference_answer"],
        generation=row["generation"],
    )

def judge_one_row(row, max_retries=4):
    prompt = build_prompt(row)
    estimated_input_tokens = estimate_tokens(prompt)

    if estimated_input_tokens > TPM_HARD:
        raise ValueError(f"Row {row['id']} estimated input exceeds TPM hard limit.")

    last_error = None
    for attempt in range(max_retries):
        wait_for_rate_limit(estimated_input_tokens)
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0,
                    response_mime_type="application/json",
                    response_schema=JUDGE_SCHEMA,
                ),
            )

            result = json.loads(response.text)

            for d in DIMENSIONS:
                if not isinstance(result.get(d), int) or not 0 <= result[d] <= 4:
                    raise ValueError(f"Invalid {d}={result.get(d)!r} for {row['id']}")

            if not reason_ok(result):
                if attempt == 0:
                    # One immediate retry: restate the reason rule inside the same rate-limit slot.
                    last_error = ValueError(
                        f"Reason-rule violation for {row['id']} (retrying once): "
                        f"scores={[result[d] for d in DIMENSIONS]} reason={result.get('reason', '')!r}"
                    )
                    continue
                raise ValueError(f"Reason-rule violation for {row['id']}: scores={[result[d] for d in DIMENSIONS]} reason={result.get('reason', '')!r}")

            return result

        except Exception as e:
            last_error = e
            if attempt == max_retries - 1:
                raise
            time.sleep(min(30 * (2 ** attempt), 120))

    raise last_error

def auto_mark_truncated(row):
    """No API call: truncated generations score 0 with a fixed reason."""
    scores = {d: 0 for d in DIMENSIONS}
    return {
        **scores,
        "reason": AUTO_REASON_TRUNCATED,
        "final_score": 0.0,
        "judged": "auto",
    }


## 7. Run evaluation: one generation per API call (+ auto-marked truncated rows)

Output is checkpointed after every row. Re-running resumes from existing ids.

In [ ]:
existing = {}
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                existing[r["id"]] = r

print(f"Existing completed rows: {len(existing)}")
print(f"Remaining rows: {len(rows) - len(existing)}")

with open(OUTPUT_FILE, "a", encoding="utf-8") as out:
    for row in tqdm(rows, desc="LLM judging"):
        if row["id"] in existing:
            continue

        if row.get("truncated") is True:
            judge = auto_mark_truncated(row)
        else:
            raw = judge_one_row(row)
            judge = {
                **{d: raw[d] for d in DIMENSIONS},
                "reason": raw.get("reason", ""),
                "final_score": compute_final_score(raw),
                "judged": "llm",
            }

        output_row = {
            "id": row["id"],
            "subject": row.get("subject", ""),
            "grade": row.get("grade", ""),
            "district": row.get("district", ""),
            "topic": row.get("topic", ""),
            "context": row.get("context", ""),
            **judge,
        }

        out.write(json.dumps(output_row, ensure_ascii=False) + "\n")
        out.flush()
        existing[row["id"]] = output_row

print(f"Done. Results saved to {OUTPUT_FILE}")


## 8. Validate judge output

In [ ]:
judge_rows = []
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            judge_rows.append(json.loads(line))

score_fields = [
    "groundedness", "faithfulness", "hallucination",
    "bengali_quality", "answer_relevancy", "instruction_following",
]
assert len(judge_rows) == len(set(r["id"] for r in judge_rows)), "Duplicate output IDs found."

for r in judge_rows:
    for field in score_fields:
        assert isinstance(r[field], int) and 0 <= r[field] <= 4, f"Invalid {field} for {r['id']}"
    # final_score must equal the recomputed mean
    expected = round(sum(r[d] for d in score_fields) / len(score_fields), 2)
    assert r["final_score"] == expected, f"Bad final_score for {r['id']}: {r['final_score']} != {expected}"
    # reason rule: empty iff all six are 4
    all_four = all(r[d] == 4 for d in score_fields)
    has_reason = bool(r.get("reason", "").strip())
    assert has_reason != all_four, f"Reason-rule violation for {r['id']}"
    assert r["judged"] in {"llm", "auto"}, f"Bad judged flag for {r['id']}"

print(f"Validated {len(judge_rows)} judge results.")
print("LLM-judged:", sum(1 for r in judge_rows if r["judged"] == "llm"))
print("Auto-marked:", sum(1 for r in judge_rows if r["judged"] == "auto"))


## 9. Summary

In [ ]:
df = pd.DataFrame(judge_rows)

summary = {f"{d}_mean": round(float(df[d].mean()), 3) for d in score_fields}
summary["final_score_mean"] = round(float(df["final_score"].mean()), 3)
summary["n_llm"] = int((df["judged"] == "llm").sum())
summary["n_auto"] = int((df["judged"] == "auto").sum())
summary["perfect_rows"] = int(((df[score_fields] == 4).all(axis=1)).sum())
summary["rows_with_reason"] = int((df["reason"].str.strip() != "").sum())

print("LLM-AS-JUDGE SUMMARY (0-4 scale)")
print("=" * 60)
for k, v in summary.items():
    print(f"{k}: {v}")

print("\nFinal score distribution:")
print(df["final_score"].value_counts().sort_index())

print("\nMean final by subject:")
print(df.groupby("subject")["final_score"].mean().round(3))

with open("/content/qwen_judge_6dim_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

df.to_csv("/content/qwen_judge_6dim_results.csv", index=False)

print("\nSaved:")
print("/content/qwen_judge_6dim_results.jsonl")
print("/content/qwen_judge_6dim_results.csv")
print("/content/qwen_judge_6dim_summary.json")
